In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 11 — OS VIZINHOS QUE DECIDEM: K-NEAREST NEIGHBORS

> "Você pode conhecer o caráter de uma pessoa pela companhia que ela mantém. Nos dados, um ponto desconhecido revela sua identidade pela proximidade de seus vizinhos."
> — Antigo ditado da Análise de Dados

Depois de explorar probabilidades, geometria e regras, eu queria algo mais orgânico. Pensei em como os patologistas trabalham: diante de uma célula difícil, eles a comparam com milhares que já viram. "Esta se parece com aquele caso benigno da semana passada." Eles se baseiam na similaridade, na vizinhança.

O **K-Nearest Neighbors (KNN)** funciona exatamente assim. É talvez o algoritmo mais simples de entender: ele não "aprende" um modelo, apenas memoriza o treino. Quando uma amostra nova chega, ele a classifica pelo voto da maioria dos seus vizinhos mais próximos. A simplicidade me atraía — mas escondia perguntas: o que significa "mais próximo" em 30 dimensões? Quantos vizinhos consultar?

## KNN: A Democracia da Vizinhança

O KNN é um algoritmo de "aprendizado preguiçoso" (*lazy learning*): não faz cálculo no treino, só armazena os dados. Na **previsão**, ele (1) calcula a distância do ponto novo a todos os pontos do treino, (2) identifica os *K* mais próximos e (3) faz a votação. Se K=5 e 3 dos 5 vizinhos são malignos, a previsão é maligno.

Duas vulnerabilidades exigem atenção:

**Escala das *features*** — o KNN depende de distância. Se `mean area` (valores nas centenas) e `mean smoothness` (valores em 0,1) convivem sem padronização, a distância é dominada pela primeira. Para o KNN, **normalizar não é opcional**.

**Maldição da dimensionalidade** — em muitas dimensões, "proximidade" perde significado; o KNN vai melhor com poucas *features* informativas.

O hiperparâmetro central é o próprio **K**. K pequeno = sensível a ruído (*overfitting*); K grande = fronteira suave demais (*underfitting*). Testamos vários por validação cruzada.

#### Código 11.1: Encontrando o Melhor K


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from oncoclassify_utils import X_train, y_train

ks = range(1, 31)
scores = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)),
                          X_train, y_train, cv=5, scoring="accuracy").mean() for k in ks]

melhor_k = list(ks)[int(np.argmax(scores))]
print(f"Melhor K = {melhor_k}  (acuracia {max(scores):.4f})")

plt.figure(figsize=(11, 6))
plt.plot(list(ks), scores, marker="o", linestyle="--")
plt.axvline(melhor_k, color="r", linestyle="--",
            label=f"Melhor K = {melhor_k} (acuracia {max(scores):.4f})")
plt.title("Desempenho do KNN vs. Valor de K"); plt.xlabel("K (nº de vizinhos)")
plt.ylabel("Acurácia média (CV)"); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

Melhor K = 5  (acuracia 0.9750)


![Desempenho do KNN vs. K](media/figura_11_1.png)

A curva tem o formato clássico: para K muito baixo, desempenho bom mas volátil (sensível a ruído); o pico ocorre em **K=5**, com **97,50%** de acurácia; depois, o desempenho decai lentamente à medida que vizinhos distantes e menos relevantes entram na votação.

Com K=5, o **recall maligno é 94,35%** — competitivo, empatando com o SVM-RBF em acurácia (97,5%) e ficando um pouco atrás em recall maligno (96,0%). Somada à sua interpretabilidade (podemos literalmente inspecionar os vizinhos de uma previsão), o KNN é uma adição valiosa ao arsenal.

> **📌 CHECKLIST DO CAPÍTULO 11**
>
> ✅ Entende o funcionamento do KNN: distância → K vizinhos → votação.
>
> ✅ Sabe por que ele é um *lazy learner*.
>
> ✅ Compreende por que a **normalização é essencial** para o KNN.
>
> ✅ Executou o Código 11.1 e encontrou **K=5** (acurácia 97,5%, recall maligno 94,4%).
>
> **⚠️ CRÍTICO:** O KNN é lento na previsão — calcula distância a *todos* os pontos de treino. Viável para nossas 480 amostras; impraticável para milhões.

O conceito de vizinhança provou-se um guia poderoso: a identidade de um ponto definida por sua companhia, simples e eficaz. Um modelo que não precisava de teorias complexas, só de uma boa régua e da capacidade de contar votos.

Faltava uma última peça no quebra-cabeça dos algoritmos fundamentais. Uma que, ironicamente, tem "regressão" no nome, mas é um dos classificadores mais populares e interpretáveis que existem. Era hora de construir a ponte entre os dois mundos: a Regressão Logística.
